# Accelerated Fine-Tuning with Unsloth

**Learning objectives:** Understand **QLoRA** with Unsloth, **Alpaca** vs **ChatML** templates, **SFTTrainer** setup, synthetic **Q&A** from documents, and **GGUF** / **Ollama** export patterns.

> Full training needs **GPU** and CUDA. This notebook is readable on CPU (configs and string demos only).

## Introduction

**Fine-tuning** continues training a pretrained LM on task data. **LoRA** adds low-rank adapters; **QLoRA** trains adapters while the base stays quantized (e.g. 4-bit), cutting VRAM. **Unsloth** speeds LoRA/QLoRA and works with **transformers** and **trl**. Match **prompt templates** to your base model for train and inference.

In [ ]:
# QLoRA-style config (print-only; no GPU required)
from pprint import pprint

cfg = {
    "max_seq_length": 2048,
    "load_in_4bit": True,
    "r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.0,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "max_steps": 60,
    "optim": "adamw_8bit",
}
print("Illustrative QLoRA / Unsloth-style kwargs:")
pprint(cfg)

try:
    import unsloth  # noqa: F401
    print("unsloth version:", getattr(unsloth, "__version__", "unknown"))
except ImportError:
    print("Install: pip install unsloth")

### Dataset formatting: Alpaca vs ChatML

**Alpaca:** instruction + optional input + response. **ChatML:** role blocks with start/end markers (exact tokens follow the model card). Training and inference must use the same template family.

In [ ]:
def format_alpaca(instruction: str, inp: str, out: str) -> str:
    if inp:
        return (
            f"### Instruction:\n{instruction}\n\n### Input:\n{inp}\n\n### Response:\n{out}"
        )
    return f"### Instruction:\n{instruction}\n\n### Response:\n{out}"


def format_chatml(system: str, user: str, assistant: str) -> str:
    return (
        f"<|im_start|>system\n{system}<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n{assistant}<|im_end|>"
    )


print(format_alpaca("Summarize.", "The cat sat.", "A cat rested.")[:120], "...")
print(format_chatml("You are helpful.", "Hi", "Hello.")[:120], "...")

### SFTTrainer setup (walkthrough)

`trl.SFTTrainer` + `SFTConfig` consume a Hugging Face **Dataset** (often a `text` column after templating). Unsloth's `FastLanguageModel.from_pretrained` + `get_peft_model` attach LoRA before training. Below: print config only; no weights downloaded.

In [ ]:
try:
    from trl import SFTConfig, SFTTrainer
except ImportError:
    print("Install: pip install trl transformers datasets accelerate peft bitsandbytes torch")
else:
    cfg = SFTConfig(
        output_dir="./unsloth_sft_out",
        max_steps=10,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=1,
        dataset_text_field="text",
    )
    for k in ("output_dir", "max_steps", "per_device_train_batch_size", "learning_rate", "dataset_text_field"):
        print(k, getattr(cfg, k, None))
    print(
        "Typical wiring (run on GPU with model + data):\n"
        "  model, tokenizer = FastLanguageModel.from_pretrained(...)\n"
        "  model = FastLanguageModel.get_peft_model(model, r=16, ...)\n"
        "  trainer = SFTTrainer(model=model, train_dataset=ds, processing_class=tokenizer, args=cfg)\n"
        "  trainer.train()"
    )

### Synthetic Q&A pairs from documents

Build instruction rows from short synthetic documents (no LLM) to test formatting and DataLoader code.

In [ ]:
documents = [
    "Project Phoenix uses Dagster for orchestration and MLflow for metrics.",
    "API rate limit: 1000 requests per hour per key.",
    "On-call rotations reset Mondays 09:00 UTC.",
]


def format_alpaca(instruction: str, inp: str, out: str) -> str:
    if inp:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{inp}\n\n### Response:\n{out}"
    return f"### Instruction:\n{instruction}\n\n### Response:\n{out}"


def doc_to_qa(doc: str, idx: int) -> dict:
    words = doc.replace(".", "").split()
    topic = " ".join(words[:3])
    q = f"What is document {idx} about in one phrase?"
    a = f"It discusses: {topic}..."
    return {"instruction": "Answer briefly.", "input": q, "output": a, "source": doc}


rows = [doc_to_qa(d, i) for i, d in enumerate(documents)]
for row in rows:
    print(format_alpaca(row["instruction"], row["input"], row["output"])[:160], "...\n")

### Export: GGUF and Ollama

**GGUF** is used by **llama.cpp** and **Ollama**. Unsloth can export merged adapters to GGUF; then point an Ollama `Modelfile` at the file. Full export needs trained weights and often GPU RAM.

In [ ]:
snippet = '''
# After training (examples only):
# model.save_pretrained_merged("merged_hf", tokenizer, save_method="merged_16bit")
# model.save_pretrained_gguf("model", tokenizer, quantization_method="q4_k_m")

# Ollama Modelfile (conceptual):
# FROM ./model-unsloth.Q4_K_M.gguf
# PARAMETER temperature 0.7
'''
print(snippet)

try:
    import unsloth  # noqa: F401
except ImportError:
    print("Install Unsloth + llama.cpp toolchain per upstream docs for GGUF export.")

## Conclusion and best practices

- Align **templates** (Alpaca, ChatML, Llama 3, etc.) for train and inference.
- Start with a **tiny** run; scale steps and data after sanity checks.
- **QLoRA** saves memory but not data quality: invest in **evaluation** and validation prompts.
- For **GGUF / Ollama**, pick quantization (e.g. Q4_K_M) for latency vs quality.

Re-run on a **GPU** machine when you are ready for `FastLanguageModel` + `SFTTrainer` end to end.